# 3. Gold Layer Business Aggregations — Line-by-Line Breakdown

This notebook transforms cleaned Silver layer tables into high-level business metrics and Gold data products, including daily revenue tracking, product category performance, and customer lifetime value (LTV).

--- 
## Cell 1: Gold Table 1 — Daily Revenue Aggregation (`gold_daily_revenue`)

In [ ]:
%sql
-- Gold #1: daily revenue, completed orders only, one row per day
CREATE OR REPLACE TABLE shopstream.core.gold_daily_revenue AS
SELECT
  DATE(order_ts) AS order_date,
  COUNT(DISTINCT order_id) AS orders,
  SUM(quantity) AS units_sold,
  ROUND(SUM(line_revenue), 2) AS revenue
FROM shopstream.core.silver_orders
WHERE status = 'completed'
GROUP BY DATE(order_ts);

SELECT * FROM shopstream.core.gold_daily_revenue ORDER BY order_date LIMIT 7

### Line-by-Line Code Explanation:

1. **`%sql`**
   - Magic command instructing Databricks to execute the cell contents as Spark SQL.

2. **`CREATE OR REPLACE TABLE shopstream.core.gold_daily_revenue AS`**
   - Atomically creates or overwrites the `gold_daily_revenue` Delta table inside Unity Catalog (`shopstream.core` schema) using the output of the query.

3. **`SELECT`**
   - Begins the column selection and aggregation definitions.

4. **`DATE(order_ts) AS order_date,`**
   - Truncates the timestamp column `order_ts` down to a calendar date (`YYYY-MM-DD`) and aliases it as `order_date`.

5. **`COUNT(DISTINCT order_id) AS orders,`**
   - Counts unique order identifiers per day (preventing multiple items in a single order from inflating order counts) and aliases it as `orders`.

6. **`SUM(quantity) AS units_sold,`**
   - Sums up the total items/units sold across all order lines for that day and aliases it as `units_sold`.

7. **`ROUND(SUM(line_revenue), 2) AS revenue`**
   - Calculates total monetary revenue for the day by summing individual line item revenues and rounding the result to two decimal places.

8. **`FROM shopstream.core.silver_orders`**
   - Identifies the cleaned `silver_orders` dataset as the source table.

9. **`WHERE status = 'completed'`**
   - Applies a business filter to exclude non-revenue states (e.g., returned or incomplete orders) and retain only completed purchases.

10. **`GROUP BY DATE(order_ts);`**
    - Aggregates the metrics so that each row in the resulting table represents exactly one calendar day.

11. **`SELECT * FROM shopstream.core.gold_daily_revenue ORDER BY order_date LIMIT 7`**
    - Executes a verification query to preview the first 7 days of daily revenue metrics in chronological order.

--- 
## Cell 2: Gold Table 2 — Category Performance & Gross Margin (`gold_category_performance`)

In [ ]:
%sql
-- Gold #2: category performance, what actually makes money
CREATE OR REPLACE TABLE shopstream.core.gold_category_performance AS
SELECT
  p.category,
  COUNT(DISTINCT o.order_id) AS orders,
  SUM(o.quantity) AS units_sold,
  ROUND(SUM(o.line_revenue), 2) AS revenue,
  ROUND(SUM(o.quantity * p.unit_margin), 2) AS gross_margin
FROM shopstream.core.silver_orders o
JOIN shopstream.core.silver_products p USING (product_id)
WHERE o.status = 'completed'
GROUP BY p.category;

SELECT * FROM shopstream.core.gold_category_performance ORDER BY revenue DESC

### Line-by-Line Code Explanation:

1. **`CREATE OR REPLACE TABLE shopstream.core.gold_category_performance AS`**
   - Defines or replaces the Gold target table focused on product category profitability.

2. **`p.category,`**
   - Selects the product category attribute from the joined product dimension table.

3. **`COUNT(DISTINCT o.order_id) AS orders,`**
   - Counts unique completed order IDs associated with each category.

4. **`SUM(o.quantity) AS units_sold,`**
   - Sums total units sold within each product category.

5. **`ROUND(SUM(o.line_revenue), 2) AS revenue,`**
   - Computes total revenue generated per category rounded to two decimal places.

6. **`ROUND(SUM(o.quantity * p.unit_margin), 2) AS gross_margin`**
   - Multiplies item units sold by the unit profit margin (`unit_price - unit_cost`) from the product dimension table, summing the values to calculate total category gross profit margin.

7. **`FROM shopstream.core.silver_orders o`**
   - Specifies the fact table `silver_orders`, aliasing it as `o`.

8. **`JOIN shopstream.core.silver_products p USING (product_id)`**
   - Performs an inner join with the `silver_products` dimension table (`p`) on the shared key `product_id`.

9. **`WHERE o.status = 'completed'`**
   - Restricts calculations to finalized, completed transactions.

10. **`GROUP BY p.category;`**
    - Groups all aggregate revenue and margin metrics by individual product categories.

11. **`SELECT * FROM shopstream.core.gold_category_performance ORDER BY revenue DESC`**
    - Previews the category performance data sorted from highest revenue to lowest.

--- 
## Cell 3: Gold Table 3 — Customer Lifetime Value Aggregation (`gold_customer_ltv`)

In [ ]:
%sql
-- Gold #3: customer lifetime value, who the best customers are
CREATE OR REPLACE TABLE shopstream.core.gold_customer_ltv AS
SELECT
  c.customer_id,
  c.name,
  c.country,
  c.signup_channel,
  COUNT(DISTINCT o.order_id) AS lifetime_orders,
  ROUND(SUM(o.line_revenue), 2) AS lifetime_revenue
FROM shopstream.core.silver_orders o
JOIN shopstream.core.silver_customers c USING (customer_id)
WHERE o.status = 'completed'
GROUP BY c.customer_id, c.name, c.country, c.signup_channel;

SELECT * FROM shopstream.core.gold_customer_ltv ORDER BY lifetime_revenue DESC LIMIT 10

### Line-by-Line Code Explanation:

1. **`CREATE OR REPLACE TABLE shopstream.core.gold_customer_ltv AS`**
   - Creates or updates the Gold aggregate table storing Customer Lifetime Value (LTV) metrics.

2. **`c.customer_id, c.name, c.country, c.signup_channel,`**
   - Selects customer demographic and acquisition attributes from the customer dimension table.

3. **`COUNT(DISTINCT o.order_id) AS lifetime_orders,`**
   - Counts the cumulative number of unique completed orders placed by each customer.

4. **`ROUND(SUM(o.line_revenue), 2) AS lifetime_revenue`**
   - Sums total monetary revenue generated by each customer over their entire lifetime.

5. **`FROM shopstream.core.silver_orders o`**
   - Sets `silver_orders` as the base order fact table (`o`).

6. **`JOIN shopstream.core.silver_customers c USING (customer_id)`**
   - Joins order records to customer profiles (`silver_customers`) matching on `customer_id`.

7. **`WHERE o.status = 'completed'`**
   - Filters for valid, completed orders.

8. **`GROUP BY c.customer_id, c.name, c.country, c.signup_channel;`**
    - Aggregates metrics per individual customer along with their descriptive metadata.

9. **`SELECT * FROM shopstream.core.gold_customer_ltv ORDER BY lifetime_revenue DESC LIMIT 10`**
    - Queries the top 10 highest-value customers based on total lifetime spend.